# 07 — Astronomical Feature Engineering

## Purpose

Notebook 06 fixed the cleaned observational foundation and the rule that
deterministic features may be evaluated at forecast-valid time. This notebook
now constructs the astronomical layers of the feature hierarchy:

- **F2:** deterministic local solar geometry;
- **F3:** continuous lunar and Sun–Moon geometry.

The dataset-provided `Azimuth[deg]` and `Elevation[deg]` columns are used only
for diagnosis. Model features are recomputed independently from timestamp,
location, and documented astronomical methods.

This notebook also fixes one important SOLETE-specific convention: how a
timestamped 5-minute row should be represented astronomically.

### References used for the astronomical method

- Pombo, Gehrke & Bindner (2022), *Data in Brief*,
  DOI `10.1016/j.dib.2022.108046` — SOLETE location and aggregation context.
- Reda & Andreas (2004), *Solar Energy*,
  DOI `10.1016/j.solener.2003.12.003` — NREL Solar Position Algorithm (SPA).
- Park et al. (2021), *The Astronomical Journal*,
  DOI `10.3847/1538-3881/abd414` — JPL DE440/DE441 ephemerides.

The objective is a transparent astronomical control, not a claim that
astronomical variables causally drive renewable generation.


In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

import pvlib
import skyfield

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete

DATA_5MIN = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5"

if not DATA_5MIN.exists():
    raise FileNotFoundError(DATA_5MIN)

df_raw = load_solete(DATA_5MIN).sort_index()

SITE_NAME = "DTU SYSLAB, Roskilde, Denmark"
SITE_LATITUDE_DEG = 55.6867
SITE_LONGITUDE_DEG = 12.0985
SITE_CIVIL_TIMEZONE = "Europe/Copenhagen"

# The SOLETE publication reports latitude/longitude but not a site altitude.
# We make pvlib's 0 m computational convention explicit instead of leaving
# it as a hidden default. F2 uses geometric rather than refracted elevation.
SOLAR_ALTITUDE_CONVENTION_M = 0.0

def solar_position(times):
    # Fixed deterministic solar-position convention used throughout this notebook.
    return pvlib.solarposition.get_solarposition(
        time=times,
        latitude=SITE_LATITUDE_DEG,
        longitude=SITE_LONGITUDE_DEG,
        altitude=SOLAR_ALTITUDE_CONVENTION_M,
        method="nrel_numpy",
        delta_t=None,
    )

def circular_difference_deg(a, b):
    # Smallest absolute separation between two angles in degrees.
    return np.abs((np.asarray(a) - np.asarray(b) + 180.0) % 360.0 - 180.0)

print("Dataset:", df_raw.shape)
print("Period:", df_raw.index.min(), "to", df_raw.index.max())
print("Timezone:", df_raw.index.tz)
print("pvlib:", pvlib.__version__)
print("Skyfield:", skyfield.__version__)


Dataset: (131617, 11)
Period: 2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00
Timezone: UTC
pvlib: 0.15.2
Skyfield: 1.55


## 1. Time and location assumptions

SOLETE reports the site at latitude $55.6867^\circ$ N and longitude
$12.0985^\circ$ E. The supplied timeline is treated as UTC.

UTC remains the canonical index because it is continuous through daylight
saving transitions. Copenhagen civil time is useful for interpretation, but
naive local wall-clock time repeats during the autumn transition.

Astronomy in this notebook is stored by **dataset row timestamp** $r$. Later,
for a forecasting sample issued at $t$ with horizon $h$,

$$
v=t+h
$$

is the target row timestamp. The deterministic astronomical row selected by
the sample builder will therefore be the row keyed by $v$, not the row keyed
by $t$.


In [2]:
local_index = df_raw.index.tz_convert(SITE_CIVIL_TIMEZONE)
local_naive = local_index.tz_localize(None)

utc_offsets = pd.Series(
    [x.utcoffset().total_seconds() / 3600 for x in local_index],
    index=df_raw.index,
    name="UTC_offset_hours",
)

offset_changes = utc_offsets.ne(utc_offsets.shift())
offset_changes.iloc[0] = False

timeline_check = pd.Series(
    {
        "chronological_UTC": df_raw.index.is_monotonic_increasing,
        "duplicate_UTC_timestamps": int(df_raw.index.duplicated().sum()),
        "repeated_naive_local_timestamps": int(local_naive.duplicated().sum()),
        "UTC_offset_transitions": int(offset_changes.sum()),
        "site_latitude_deg": SITE_LATITUDE_DEG,
        "site_longitude_deg": SITE_LONGITUDE_DEG,
        "solar_altitude_computational_convention_m":
            SOLAR_ALTITUDE_CONVENTION_M,
    },
    name="astronomical assumptions",
)

display(timeline_check.to_frame("value"))


,value
chronological_UTC,True
duplicate_UTC_timestamps,0
repeated_naive_local_timestamps,12
UTC_offset_transitions,2
site_latitude_deg,55.6867
site_longitude_deg,12.0985
solar_altitude_computational_convention_m,0.0


## 2. Why the supplied solar-position columns remain diagnostic only

Earlier EDA showed that the source `Azimuth[deg]` and `Elevation[deg]` columns
do not behave like a conventional signed solar-position pair.

We test two questions without modifying either source column:

1. Is the azimuth approximately a South-referenced signed angle?
2. Is the residual disagreement associated with a simple clock offset?

For azimuth, the diagnostic conversion

$$
a_{\mathrm{north}}=(a_{\mathrm{source}}+180^\circ)\bmod360^\circ
$$

maps a South-referenced convention onto pvlib's North-referenced
$[0^\circ,360^\circ)$ convention.

The purpose is not to recover an undocumented upstream algorithm. It is only
to decide whether the source columns are trustworthy enough to become F2
features.


In [3]:
source_mask = df_raw["Elevation[deg]"] > 0
source_times = df_raw.index[source_mask]

source_elevation = df_raw.loc[source_mask, "Elevation[deg]"].to_numpy()
source_azimuth_north = (
    (df_raw.loc[source_mask, "Azimuth[deg]"] + 180.0) % 360.0
).to_numpy()

source_offsets = utc_offsets.loc[source_mask].to_numpy()

candidate_times = {
    "recorded UTC": source_times,
    "-1 hour": source_times - pd.Timedelta(hours=1),
    "-2 hours": source_times - pd.Timedelta(hours=2),
    "- Copenhagen civil offset":
        source_times - pd.to_timedelta(source_offsets, unit="h"),
}

rows = []
for label, times in candidate_times.items():
    pos = solar_position(times)
    rows.append(
        {
            "time_interpretation": label,
            "elevation_MAE_deg":
                float(np.abs(pos["elevation"].to_numpy() - source_elevation).mean()),
            "azimuth_MAE_deg":
                float(
                    circular_difference_deg(
                        pos["azimuth"].to_numpy(),
                        source_azimuth_north,
                    ).mean()
                ),
        }
    )

source_solar_diagnostic = (
    pd.DataFrame(rows)
    .sort_values("elevation_MAE_deg")
    .reset_index(drop=True)
)

print("Positive source-elevation rows:", int(source_mask.sum()))
display(source_solar_diagnostic)


Positive source-elevation rows: 70572


,time_interpretation,elevation_MAE_deg,azimuth_MAE_deg
0,-1 hour,0.621785,3.181073
1,- Copenhagen civil offset,5.294609,10.602857
2,recorded UTC,5.601426,18.275194
3,-2 hours,6.210183,11.943579


### Source-column decision

The fixed one-hour offset gives substantially better agreement than the other
tested timestamp interpretations after converting the source azimuth from its
South-referenced convention to pvlib's North-referenced convention.

This pattern is useful evidence that the supplied solar-position columns use a
different representation/time convention. It is **not** evidence that the
SOLETE measurement timeline itself should be shifted.

Accordingly, F2 will be calculated independently from the verified UTC
timeline and site coordinates.

## 3. Choosing the astronomical representative time of a 5-minute row

The SOLETE data paper states that higher-resolution measurements were averaged
to obtain the 5-minute product. Notebook 05 additionally found strong numerical
consistency between the legacy 5-minute values and means of the five 1-minute
samples labelled

`r`, `r+1 min`, `r+2 min`, `r+3 min`, and `r+4 min`.

This is an **empirical aggregation-consistency result**; it does not claim
knowledge of an undocumented upstream processing implementation.

We therefore compare three deterministic representatives of row $r$:

- recorded timestamp: $r$;
- centre of the five discrete 1-minute samples: $r+2$ minutes;
- midpoint of the continuous interval $[r,r+5\text{ min})$: $r+2.5$ minutes.

The reference is the mean independently computed solar state over the five
1-minute timestamps. No measured weather or power value is used in this test.


In [4]:
astronomy_1min_times = pd.date_range(
    start=df_raw.index.min(),
    end=df_raw.index.max(),
    freq="1min",
    tz="UTC",
)

solar_1min = solar_position(astronomy_1min_times)

elevation_mean_5min = (
    solar_1min["elevation"]
    .resample("5min", label="left", closed="left")
    .mean()
    .reindex(df_raw.index)
)

az_rad = np.deg2rad(solar_1min["azimuth"])

az_sin_mean = (
    pd.Series(np.sin(az_rad), index=astronomy_1min_times)
    .resample("5min", label="left", closed="left")
    .mean()
    .reindex(df_raw.index)
)

az_cos_mean = (
    pd.Series(np.cos(az_rad), index=astronomy_1min_times)
    .resample("5min", label="left", closed="left")
    .mean()
    .reindex(df_raw.index)
)

azimuth_mean_5min = np.rad2deg(
    np.arctan2(az_sin_mean, az_cos_mean)
) % 360.0

interval_counts = (
    pd.Series(1, index=astronomy_1min_times)
    .resample("5min", label="left", closed="left")
    .count()
    .reindex(df_raw.index)
)
complete_bin = interval_counts.eq(5)

candidate_row_times = {
    "r": df_raw.index,
    "r+2min": df_raw.index + pd.Timedelta(minutes=2),
    "r+2.5min": df_raw.index + pd.Timedelta(minutes=2.5),
}

rows = []
for label, times in candidate_row_times.items():
    pos = solar_position(times)
    pos.index = df_raw.index

    elevation_error = (
        pos["elevation"] - elevation_mean_5min
    ).abs()

    azimuth_error = pd.Series(
        circular_difference_deg(pos["azimuth"], azimuth_mean_5min),
        index=df_raw.index,
    )

    rows.append(
        {
            "representative_time": label,
            "elevation_MAE_deg":
                float(elevation_error.loc[complete_bin].mean()),
            "elevation_max_error_deg":
                float(elevation_error.loc[complete_bin].max()),
            "azimuth_MAE_deg":
                float(azimuth_error.loc[complete_bin].mean()),
            "azimuth_max_error_deg":
                float(azimuth_error.loc[complete_bin].max()),
        }
    )

representative_time_comparison = pd.DataFrame(rows)
display(representative_time_comparison)


,representative_time,elevation_MAE_deg,elevation_max_error_deg,azimuth_MAE_deg,azimuth_max_error_deg
0,r,0.190628,0.282372,0.500004,0.859457
1,r+2min,0.000391,0.001057,0.000327,0.001197
2,r+2.5min,0.047659,0.070595,0.125001,0.214885


### SOLETE astronomical time convention

The expected result is that $r+2$ minutes is almost exactly the single-time
representation of the five discrete 1-minute astronomical samples.

We therefore define, for each SOLETE row timestamp $r$,

$$
r_{\mathrm{astro}}=r+2\text{ min}.
$$

This offset belongs to the **row representation**, not to the forecast issue
time.

For a model issued at $t$ with valid row $v=t+h$, F2–F4 will consequently be
selected from row $v$ and represent astronomy at

$$
v_{\mathrm{astro}}=t+h+2\text{ min}.
$$

This distinction prevents the easy mistake of attaching F2–F4 to issue time.

The $+2$ minute rule is SOLETE-specific and must not be transferred
automatically to another dataset.


In [5]:
astro_reference_times = df_raw.index + pd.Timedelta(minutes=2)

solar_final = solar_position(astro_reference_times)
solar_final.index = df_raw.index

df_solar_features = pd.DataFrame(index=df_raw.index)
df_solar_features["solar_elevation_deg"] = solar_final["elevation"]

zenith_rad = np.deg2rad(solar_final["zenith"])
df_solar_features["solar_cos_zenith"] = np.cos(zenith_rad)

azimuth_rad = np.deg2rad(solar_final["azimuth"])
df_solar_features["solar_azimuth_sin"] = np.sin(azimuth_rad)
df_solar_features["solar_azimuth_cos"] = np.cos(azimuth_rad)

df_solar_features["solar_daylight"] = (
    df_solar_features["solar_elevation_deg"] > 0
).astype("int8")

azimuth_norm_error = np.abs(
    df_solar_features["solar_azimuth_sin"] ** 2
    + df_solar_features["solar_azimuth_cos"] ** 2
    - 1
).max()

print("F2 shape:", df_solar_features.shape)
print("Missing F2 values:", int(df_solar_features.isna().sum().sum()))
print("Maximum azimuth unit-circle error:", float(azimuth_norm_error))
display(df_solar_features.describe().T)


F2 shape: (131617, 5)
Missing F2 values: 0
Maximum azimuth unit-circle error: 2.220446049250313e-16


,count,mean,std,min,25%,50%,75%,max
solar_elevation_deg,131617.0,3.750422,28.134791,-57.747677,-15.273935,3.192199,24.907054,57.746609
solar_cos_zenith,131617.0,0.058648,0.447715,-0.845706,-0.263434,0.055686,0.421147,0.845696
solar_azimuth_sin,131617.0,-0.000013,0.740039,-1.000000,-0.771034,-0.000085,0.771015,1.000000
solar_azimuth_cos,131617.0,0.024824,0.672111,-1.000000,-0.607760,0.043187,0.663657,1.000000
solar_daylight,131617.0,0.537507,0.498593,0.000000,0.000000,1.000000,1.000000,1.000000


## 4. Physical validation of F2

F2 should be deterministic but still physically consistent with the measured
solar-energy cycle.

For validation only, we compare same-row solar geometry with GHI, POA
irradiance, and PV power. These observed quantities are **not** being treated
as future-known forecasting inputs.

The daylight check is also descriptive. Small positive measured irradiance or
PV power close to a geometric horizon is not automatically an error because
the SOLETE row is an averaged interval and diffuse/refraction/measurement
effects remain possible.


In [6]:
solar_validation = pd.concat(
    [
        df_solar_features[
            ["solar_elevation_deg", "solar_cos_zenith", "solar_daylight"]
        ],
        df_raw[["GHI[kW1m2]", "POA Irr[kW1m2]", "P_Solar[kW]"]],
    ],
    axis=1,
)

corr = solar_validation[
    [
        "solar_elevation_deg",
        "solar_cos_zenith",
        "GHI[kW1m2]",
        "POA Irr[kW1m2]",
        "P_Solar[kW]",
    ]
].corr()

correlation_summary = pd.DataFrame(
    {
        "with_solar_elevation":
            corr.loc[
                "solar_elevation_deg",
                ["GHI[kW1m2]", "POA Irr[kW1m2]", "P_Solar[kW]"],
            ],
        "with_cos_zenith":
            corr.loc[
                "solar_cos_zenith",
                ["GHI[kW1m2]", "POA Irr[kW1m2]", "P_Solar[kW]"],
            ],
    }
)

night = df_solar_features["solar_daylight"].eq(0)

night_summary = pd.Series(
    {
        "night_rows": int(night.sum()),
        "mean_GHI_kWm2": float(df_raw.loc[night, "GHI[kW1m2]"].mean()),
        "mean_POA_kWm2": float(df_raw.loc[night, "POA Irr[kW1m2]"].mean()),
        "mean_P_Solar_kW": float(df_raw.loc[night, "P_Solar[kW]"].mean()),
        "max_P_Solar_kW": float(df_raw.loc[night, "P_Solar[kW]"].max()),
    },
    name="geometric-night diagnostic",
)

display(correlation_summary)
display(night_summary.to_frame("value"))


,with_solar_elevation,with_cos_zenith
GHI[kW1m2],0.774692,0.769514
POA Irr[kW1m2],0.692185,0.688214
P_Solar[kW],0.688781,0.685898


,value
night_rows,60872.000000
mean_GHI_kWm2,0.000079
mean_POA_kWm2,0.000167
mean_P_Solar_kW,0.001283
max_P_Solar_kW,0.191199


## 5. F3: continuous lunar and Sun–Moon astronomy

F3 must be a strong **continuous astronomical control** before categorical
Panchang variables are introduced.

We use Skyfield with JPL **DE440s**. DE440s covers the complete 2018–2019
SOLETE interval and supplies the Earth, Moon, and Sun ephemerides required
here.

Skyfield's `ecliptic_frame` is the **true ecliptic and equinox of date**.
The resulting apparent geocentric longitudes are therefore tropical/equinox-of-date
quantities at this stage.

The continuous Moon–Sun phase variable is the ecliptic-longitude difference

$$
\phi =
(\lambda_{\mathrm{Moon}}-\lambda_{\mathrm{Sun}})
\bmod 360^\circ.
$$

This is not the same thing as a generic three-dimensional angular separation.

No sidereal conversion, ayanamsha, Tithi, Paksha, Karana, Nakshatra, or Yoga
is introduced in F3.


In [7]:
from skyfield.api import Loader
from skyfield.framelib import ecliptic_frame
from skyfield import almanac

EPHEMERIS_DIR = PROJECT_ROOT / "ephemeris"
EPHEMERIS_DIR.mkdir(parents=True, exist_ok=True)

loader = Loader(str(EPHEMERIS_DIR))
ts = loader.timescale()
eph = loader("de440s.bsp")

earth = eph["Earth"]
moon = eph["Moon"]
sun = eph["Sun"]

ephemeris_file = EPHEMERIS_DIR / "de440s.bsp"

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

print("DE440s exists:", ephemeris_file.exists())
print("DE440s size [MB]:", round(ephemeris_file.stat().st_size / 1024**2, 3))
print("DE440s SHA256:", sha256_file(ephemeris_file))

sample_positions = np.linspace(
    0, len(astro_reference_times) - 1, 5, dtype=int
)
sample_times_pd = astro_reference_times[sample_positions]
sample_times_sf = ts.from_datetimes(sample_times_pd.to_pydatetime())

e = earth.at(sample_times_sf)
m = e.observe(moon).apparent()
s = e.observe(sun).apparent()

moon_lat, moon_lon, _ = m.frame_latlon(ecliptic_frame)
_, sun_lon, _ = s.frame_latlon(ecliptic_frame)

phase_from_longitudes = (
    moon_lon.degrees - sun_lon.degrees
) % 360.0

phase_from_almanac = almanac.moon_phase(
    eph, sample_times_sf
).degrees

phase_check_error = circular_difference_deg(
    phase_from_longitudes,
    phase_from_almanac,
)

print(
    "Maximum sample phase-identity error [deg]:",
    float(np.max(phase_check_error)),
)


DE440s exists: True
DE440s size [MB]: 31.21
DE440s SHA256: c1c7feeab882263fc493a9d5a5b2ddd71b54826cdf65d8d17a76126b260a49f2
Maximum sample phase-identity error [deg]: 2.842170943040401e-14


## 6. F3 feature design

The audit geometry retains six raw continuous quantities:

- Moon–Sun phase angle;
- apparent geocentric lunar ecliptic longitude;
- apparent geocentric lunar ecliptic latitude;
- apparent geocentric solar ecliptic longitude;
- lunar illuminated fraction;
- Earth–Moon distance.

The three circular angles supplied to a forecasting model are encoded with

$$
C(\theta)=
\left[
\sin\left(\frac{\pi\theta}{180}\right),
\cos\left(\frac{\pi\theta}{180}\right)
\right].
$$

The resulting F3 augmentation contains nine scalar variables:

`moon_phase_sin`, `moon_phase_cos`,
`moon_longitude_sin`, `moon_longitude_cos`,
`sun_ecliptic_longitude_sin`, `sun_ecliptic_longitude_cos`,
`moon_ecliptic_latitude_deg`, `moon_illumination_fraction`,
and `earth_moon_distance_km`.

Solar ecliptic longitude is retained even though F2 already contains local
solar geometry because F3 should explicitly control for the continuous
Sun–Moon state from which several later Panchang categories are derived.


In [8]:
skyfield_full_times = ts.from_datetimes(
    astro_reference_times.to_pydatetime()
)

earth_at_t = earth.at(skyfield_full_times)
moon_apparent = earth_at_t.observe(moon).apparent()
sun_apparent = earth_at_t.observe(sun).apparent()

moon_lat, moon_lon, moon_distance = (
    moon_apparent.frame_latlon(ecliptic_frame)
)
_, sun_lon, _ = sun_apparent.frame_latlon(ecliptic_frame)

phase_deg = (
    moon_lon.degrees - sun_lon.degrees
) % 360.0

illumination = moon_apparent.fraction_illuminated(sun)

df_lunar_geometry = pd.DataFrame(
    {
        "moon_phase_angle_deg": phase_deg,
        "moon_ecliptic_longitude_deg": moon_lon.degrees,
        "moon_ecliptic_latitude_deg": moon_lat.degrees,
        "sun_ecliptic_longitude_deg": sun_lon.degrees,
        "moon_illumination_fraction": illumination,
        "earth_moon_distance_km": moon_distance.km,
    },
    index=df_raw.index,
)

df_lunar_features = pd.DataFrame(index=df_raw.index)

for raw_col, prefix in [
    ("moon_phase_angle_deg", "moon_phase"),
    ("moon_ecliptic_longitude_deg", "moon_longitude"),
    ("sun_ecliptic_longitude_deg", "sun_ecliptic_longitude"),
]:
    angle_rad = np.deg2rad(df_lunar_geometry[raw_col])
    df_lunar_features[f"{prefix}_sin"] = np.sin(angle_rad)
    df_lunar_features[f"{prefix}_cos"] = np.cos(angle_rad)

df_lunar_features["moon_ecliptic_latitude_deg"] = (
    df_lunar_geometry["moon_ecliptic_latitude_deg"]
)
df_lunar_features["moon_illumination_fraction"] = (
    df_lunar_geometry["moon_illumination_fraction"]
)
df_lunar_features["earth_moon_distance_km"] = (
    df_lunar_geometry["earth_moon_distance_km"]
)

circle_pairs = [
    ("moon_phase_sin", "moon_phase_cos"),
    ("moon_longitude_sin", "moon_longitude_cos"),
    ("sun_ecliptic_longitude_sin", "sun_ecliptic_longitude_cos"),
]

circle_errors = {
    f"{s}/{c}":
        float(
            np.abs(
                df_lunar_features[s] ** 2
                + df_lunar_features[c] ** 2
                - 1
            ).max()
        )
    for s, c in circle_pairs
}

print("Raw F3 geometry shape:", df_lunar_geometry.shape)
print("Model-ready F3 augmentation shape:", df_lunar_features.shape)
print("Missing F3 values:", int(df_lunar_features.isna().sum().sum()))
print("Maximum circular-norm errors:")
display(pd.Series(circle_errors, name="max_error").to_frame())

display(df_lunar_geometry.describe().T)


Raw F3 geometry shape: (131617, 6)
Model-ready F3 augmentation shape: (131617, 9)
Missing F3 values: 0
Maximum circular-norm errors:


,max_error
moon_phase_sin/moon_phase_cos,2.220446e-16
moon_longitude_sin/moon_longitude_cos,2.220446e-16
sun_ecliptic_longitude_sin/sun_ecliptic_longitude_cos,2.220446e-16


,count,mean,std,min,25%,50%,75%,max
moon_phase_angle_deg,131617.0,182.356557,103.922673,0.000493,92.275431,184.943773,272.226139,359.999646
moon_ecliptic_longitude_deg,131617.0,183.568566,106.909878,0.002773,88.414626,186.270490,279.619243,359.998512
moon_ecliptic_latitude_deg,131617.0,-0.146825,3.639117,-5.301922,-3.769174,-0.282753,3.466905,5.293951
sun_ecliptic_longitude_deg,131617.0,165.324547,96.801736,0.002462,90.516633,145.104659,244.334604,359.999014
moon_illumination_fraction,131617.0,0.499584,0.351087,0.000031,0.149731,0.500789,0.848172,0.999999
earth_moon_distance_km,131617.0,384955.273706,15335.000134,356762.768577,370887.432409,385908.405712,400049.662800,406559.424079


## 7. Final astronomical protocol

The astronomical feature tables are now target-agnostic and can be used for
both PV and wind forecasting.

| Layer | Variables added | Row-time interpretation |
|---|---:|---|
| F2 | 5 solar-geometry variables | row timestamp $r+2$ min |
| F3 | 9 continuous lunar/Sun–Moon variables | row timestamp $r+2$ min |

For a forecasting sample with issue time $t$ and valid row $v=t+h$, the later
sample builder must select the deterministic row at $v$. Thus the SOLETE
astronomical calculation represented in that row is at $v+2$ minutes.

No observed future irradiance, weather, PV power, or wind power enters F2 or
F3. No scaler is fitted here.


In [9]:
verification = pd.Series(
    {
        "rows": len(df_raw),
        "F2_variables": df_solar_features.shape[1],
        "F3_variables_added": df_lunar_features.shape[1],
        "F2_missing": int(df_solar_features.isna().sum().sum()),
        "F3_missing": int(df_lunar_features.isna().sum().sum()),
        "F2_index_matches_SOLETE":
            bool(df_solar_features.index.equals(df_raw.index)),
        "F3_index_matches_SOLETE":
            bool(df_lunar_features.index.equals(df_raw.index)),
        "astronomical_row_offset_minutes": 2,
    },
    name="astronomical feature verification",
)

display(verification.to_frame("value"))

assert df_solar_features.shape[1] == 5
assert df_lunar_features.shape[1] == 9
assert verification["F2_missing"] == 0
assert verification["F3_missing"] == 0
assert verification["F2_index_matches_SOLETE"]
assert verification["F3_index_matches_SOLETE"]


,value
rows,131617
F2_variables,5
F3_variables_added,9
F2_missing,0
F3_missing,0
F2_index_matches_SOLETE,True
F3_index_matches_SOLETE,True
astronomical_row_offset_minutes,2


## Next step

F2 and F3 now provide independently calculated continuous astronomy.

The next stage can construct F4 Panchang categories while keeping the
representation boundary explicit:

- Tithi and Karana from Moon–Sun elongation;
- Paksha from Tithi phase;
- Nakshatra and Yoga only after a documented sidereal conversion and
  ayanamsha convention are validated.

That next notebook should compare the implementation against an independent
reference at selected timestamps before generating the full categorical
feature table.
